# Byte-level BPE 한국어 깨짐 분석과 해결

**자연어정보분석 | 연세대학교 글로벌인재대학**

GPT-4의 Byte-level BPE에서 한국어/이모지가 바이트 조각으로 깨지는 원인을 분석하고,
한국어 LLM들(KoGPT, EXAONE, SOLAR)이 사용하는 **방법 1+3 조합**으로 해결합니다.

| 단계 | 내용 |
|------|------|
| 1 | 왜 깨지나? — UTF-8 바이트 분해 메커니즘 |
| 2 | 방법 1: 한국어 말뭉치로 BBPE 직접 학습 |
| 3 | 방법 2: 형태소 전처리 + BBPE (하이브리드) |
| 4 | 방법 3: 한글 음절 11,172자를 초기 어휘에 추가 |
| 5 | **방법 1+3 조합** — 한국어 LLM 실전 방식 |
| 6 | 실제 한국어 LLM 토크나이저 비교 |

In [ ]:
# 환경 설정
!pip install -q tiktoken kiwipiepy tokenizers transformers

## 1. 왜 깨지나? — GPT-4 Byte-level BPE의 한국어 바이트 분해

In [ ]:
import tiktoken

enc = tiktoken.encoding_for_model("gpt-4")

def show_byte_breakdown(text):
    """텍스트의 UTF-8 바이트와 BPE 토큰 바이트를 비교"""
    print(f'\n원문: "{text}"')
    print(f'UTF-8: {text.encode("utf-8").hex(" ")}  ({len(text.encode("utf-8"))}바이트)')
    
    tokens = enc.encode(text)
    print(f'BPE:   {len(tokens)}토큰')
    
    broken = 0
    for t in tokens:
        raw = enc.decode_single_token_bytes(t)
        try:
            s = raw.decode("utf-8")
            status = f"✓ '{s}'"
        except UnicodeDecodeError:
            s = f"0x{raw.hex()}"
            status = f"✗ {s} (불완전 바이트)"
            broken += 1
        print(f'  ID {t:>6d}: [{raw.hex(" ")}] ({len(raw)}B) → {status}')
    
    if broken:
        print(f'  ⚠ {broken}개 토큰이 단독 디코딩 불가 (바이트 조각)')
    return len(tokens), broken

# 영어는 깨지지 않음 — ASCII 1바이트 = 1토큰 단위로 충분
show_byte_breakdown("Hello World")

# 한국어는 깨짐 — 3바이트 글자가 바이트 조각으로 분리
show_byte_breakdown("인공지능")

# 이모지는 더 심함 — 4바이트
show_byte_breakdown("🤖🌍✨")

In [ ]:
# 한글 한 글자가 바이트로 어떻게 쪼개지는지 시각화
print("=== 한글 UTF-8 바이트 구조 ===")
print()
for char in "가나다라인공지능":
    b = char.encode("utf-8")
    tokens = enc.encode(char)
    parts = []
    for t in tokens:
        raw = enc.decode_single_token_bytes(t)
        try:    parts.append(f"'{raw.decode('utf-8')}'")
        except: parts.append(f"0x{raw.hex()}")
    
    print(f"  '{char}' = {b.hex(' ')} (3바이트) → GPT-4: {len(tokens)}토큰 {parts}")

print()
print("→ GPT-4는 영어 중심 말뭉치로 학습되어 한글 바이트 쌍의 병합 우선순위가 낮음")
print("→ 결과: 한글 1글자(3바이트)가 2~3토큰으로 분해")

## 2. 방법 1 — 한국어 말뭉치로 Byte-level BPE 학습

한국어 텍스트로 학습하면 한글 바이트 쌍(`\xec\x9d\xb8` 등)의 빈도가 높아져 자연스럽게 병합됩니다.

In [ ]:
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders
import tempfile, os

# ── Byte-level 토큰을 사람이 읽을 수 있게 변환하는 헬퍼 ──
# ByteLevel pre_tokenizer는 바이트를 GPT-2 스타일 유니코드로 매핑함
# 예: 0xEC → 'ì', 0x9D → 'Ŀ' ... → 사람이 읽을 수 없음
# 이 매핑을 역변환하여 원래 UTF-8 문자열로 복원

def _build_byte_decoder():
    """GPT-2 byte-to-unicode 매핑의 역변환 테이블 생성"""
    # tokenizers 라이브러리의 ByteLevel.alphabet()과 동일한 매핑
    bs = list(range(ord("!"), ord("~")+1)) + list(range(ord("¡"), ord("¬")+1)) + list(range(ord("®"), ord("ÿ")+1))
    cs = bs[:]
    n = 0
    for b in range(256):
        if b not in bs:
            bs.append(b)
            cs.append(256 + n)
            n += 1
    return {chr(c): b for b, c in zip(bs, cs)}

_BYTE_DECODER = _build_byte_decoder()

def readable_tokens(token_list):
    """Byte-level BPE 토큰 리스트를 사람이 읽을 수 있는 문자열 리스트로 변환"""
    result = []
    for tok in token_list:
        raw_bytes = bytes([_BYTE_DECODER.get(c, ord(c)) for c in tok])
        try:
            result.append(raw_bytes.decode("utf-8"))
        except UnicodeDecodeError:
            result.append(f"[{raw_bytes.hex()}]")
    return result

# ── 한국어 학습 말뭉치 (실제로는 수GB 규모 사용) ──
ko_corpus = [
    "인공지능이 자연어 처리 기술을 빠르게 발전시키고 있다",
    "딥러닝 모델은 대규모 데이터에서 패턴을 학습한다",
    "자연어 처리는 컴퓨터가 사람의 언어를 이해하는 기술이다",
    "트랜스포머는 셀프어텐션 메커니즘을 사용한다",
    "토큰화는 텍스트를 작은 단위로 분리하는 과정이다",
    "한국어는 교착어로 조사와 어미가 결합하는 특징이 있다",
    "형태소 분석은 의미의 최소 단위로 분리하는 작업이다",
    "기계번역 시스템은 인코더와 디코더 구조를 활용한다",
    "언어 모델은 다음 토큰의 확률 분포를 예측한다",
    "어텐션 메커니즘은 입력 시퀀스의 관련 부분에 집중한다",
    "사전학습된 모델을 파인튜닝하여 특정 태스크에 적용한다",
    "토크나이저의 어휘 크기는 모델 성능에 큰 영향을 미친다",
] * 200  # 빈도 확보를 위해 반복

corpus_path = os.path.join(tempfile.gettempdir(), "ko_corpus.txt")
with open(corpus_path, "w") as f:
    f.write("\n".join(ko_corpus))

# ── 방법 1: 기본 Byte-level BPE (한국어 말뭉치) ──
tok_v1 = Tokenizer(models.BPE())
tok_v1.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
tok_v1.decoder = decoders.ByteLevel()

trainer_v1 = trainers.BpeTrainer(
    vocab_size=4000,
    min_frequency=2,
    special_tokens=["<pad>", "<s>", "</s>", "<unk>"],
    initial_alphabet=pre_tokenizers.ByteLevel.alphabet(),
)
tok_v1.train([corpus_path], trainer_v1)

# ── 비교 ──
test = "인공지능이 자연어 처리 기술을 발전시키고 있다"
gpt4_tokens = enc.encode(test)
v1_result = tok_v1.encode(test)
v1_readable = readable_tokens(v1_result.tokens)

print(f'원문: "{test}"')
print(f'\nGPT-4 BBPE:    {len(gpt4_tokens):>2d}토큰')
print(f'방법1 (한국어): {len(v1_result.tokens):>2d}토큰')
print(f'  내부 표현: {v1_result.tokens}')
print(f'  읽기 변환: {v1_readable}')
print(f'\n→ 한국어 말뭉치로 학습하면 한글 바이트 쌍이 자연 병합')
print(f'→ 내부적으로는 GPT-2 바이트 매핑 문자를 사용하지만, 디코딩하면 정상 한글')

## 3. 방법 2 — 형태소 전처리 + Byte-level BPE (하이브리드)

형태소 분석으로 의미 단위를 먼저 잡아주면, 기존 BBPE라도 더 나은 결과를 얻습니다.

In [ ]:
from kiwipiepy import Kiwi

kw = Kiwi()

def morpheme_pretokenize(text):
    """형태소 경계에 공백을 삽입하여 BBPE가 형태소 내부만 병합하도록 유도"""
    return " ".join(t.form for t in kw.tokenize(text))

test = "인공지능이 자연어 처리 기술을 발전시키고 있다"
pretokenized = morpheme_pretokenize(test)

raw_tokens = enc.encode(test)
pre_tokens = enc.encode(pretokenized)

print(f'원문:        "{test}"')
print(f'형태소전처리: "{pretokenized}"')
print(f'\n직접 GPT-4 BBPE: {len(raw_tokens)}토큰')
print(f'형태소 + BBPE:    {len(pre_tokens)}토큰')
print(f'\n→ 토큰 수는 비슷하지만, 형태소 경계를 존중하여 의미 단위가 보존됨')
print(f'→ 실제 활용: 형태소 전처리 후 학습하면 downstream task 성능 향상')

## 4. 방법 3 — 한글 음절 11,172자를 초기 어휘에 추가

유니코드 한글 음절 블록(`가`~`힣`, U+AC00~U+D7A3)을 BPE 학습 전에 base vocabulary에 넣으면,
한글이 **처음부터 단일 토큰**으로 존재하여 바이트 분해가 원천 차단됩니다.

In [ ]:
# 한글 음절 전체 (가~힣)
hangul_syllables = [chr(c) for c in range(0xAC00, 0xD7A4)]  # 11,172자
print(f"한글 음절 수: {len(hangul_syllables)}")
print(f"처음 20자: {''.join(hangul_syllables[:20])}")
print(f"마지막 20자: {''.join(hangul_syllables[-20:])}")

# 각 음절의 UTF-8 바이트 표현
print(f"\n각 음절 = 3바이트:")
for ch in "가나다힣":
    print(f"  '{ch}' → {ch.encode('utf-8').hex(' ')}")

print(f"\n→ 11,172 × 3바이트 = base vocab에 추가할 병합 규칙")

In [ ]:
# 방법 3: 한글 초기 어휘 + Byte-level BPE
tok_v3 = Tokenizer(models.BPE())
tok_v3.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
tok_v3.decoder = decoders.ByteLevel()

# 기본 256 바이트 + 한글 음절의 바이트 시퀀스를 초기 알파벳에 추가
base_alphabet = list(pre_tokenizers.ByteLevel.alphabet())

# 한글 바이트를 latin-1로 변환하여 초기 알파벳에 추가
hangul_byte_chars = set()
for ch in hangul_syllables:
    for b in ch.encode("utf-8"):
        hangul_byte_chars.add(chr(b))  # 바이트를 문자로

initial_alphabet = list(set(base_alphabet) | hangul_byte_chars)

trainer_v3 = trainers.BpeTrainer(
    vocab_size=16000,  # 한글 음절 수용을 위해 더 큰 어휘
    min_frequency=2,
    special_tokens=["<pad>", "<s>", "</s>", "<unk>"],
    initial_alphabet=initial_alphabet,
)
tok_v3.train([corpus_path], trainer_v3)

# 비교
test = "인공지능"
v3_result = tok_v3.encode(test)
gpt4_t = enc.encode(test)

print(f'원문: "{test}"')
print(f'GPT-4:           {len(gpt4_t)}토큰')
print(f'방법3 (초기어휘): {len(v3_result.tokens)}토큰 → {readable_tokens(v3_result.tokens)}')

## 5. 방법 1+3 조합 — 한국어 LLM 실전 방식

실제 한국어 LLM들이 사용하는 방식:
1. **한글 음절 11,172자**를 초기 어휘에 등록 → 바이트 분해 원천 차단
2. **한국어 대규모 말뭉치**로 BPE 학습 → 자주 쓰는 한국어 서브워드 자동 병합

결과: `인공지능` → `['인공', '지능']` (2토큰, GPT-4는 6~8토큰)

In [ ]:
# ── 방법 1+3 조합: 한글 초기 어휘 + 한국어 말뭉치 학습 ──

tok_combined = Tokenizer(models.BPE())
tok_combined.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
tok_combined.decoder = decoders.ByteLevel()

# 초기 알파벳: 256 base bytes + 한글 바이트
base_alphabet = list(pre_tokenizers.ByteLevel.alphabet())
hangul_byte_chars = set()
for ch in hangul_syllables:
    for b in ch.encode("utf-8"):
        hangul_byte_chars.add(chr(b))
combined_alphabet = list(set(base_alphabet) | hangul_byte_chars)

trainer_combined = trainers.BpeTrainer(
    vocab_size=32000,       # 실제 LLM 수준 어휘 크기
    min_frequency=2,
    special_tokens=["<pad>", "<s>", "</s>", "<unk>", "<mask>"],
    initial_alphabet=combined_alphabet,
)

# 한국어 말뭉치로 학습 (방법 1)
tok_combined.train([corpus_path], trainer_combined)

vocab_size = tok_combined.get_vocab_size()
print(f"어휘 크기: {vocab_size}")
print(f"한글 음절 기반 토큰 수: {sum(1 for t in tok_combined.get_vocab() if any(0xAC00 <= ord(c) <= 0xD7A3 for c in t if len(c)==1))}")

In [ ]:
# ── 최종 비교: GPT-4 vs 방법1 vs 방법3 vs 방법1+3 ──

test_sentences = [
    "인공지능이 자연어 처리 기술을 발전시키고 있다",
    "딥러닝 모델은 대규모 데이터에서 패턴을 학습한다",
    "트랜스포머는 셀프어텐션 메커니즘을 사용한다",
    "한국어는 교착어로 조사와 어미가 결합한다",
    "Hello World 인공지능 🤖",
]

print(f'{"문장":^40s} {"GPT-4":>6s} {"방법1":>6s} {"방법3":>6s} {"1+3":>6s}')
print("─" * 70)

all_data = []
for sent in test_sentences:
    g = len(enc.encode(sent))
    v1 = len(tok_v1.encode(sent).tokens)
    v3 = len(tok_v3.encode(sent).tokens)
    vc = len(tok_combined.encode(sent).tokens)
    label = sent[:38] + ".." if len(sent) > 38 else sent
    print(f'{label:40s} {g:>6d} {v1:>6d} {v3:>6d} {vc:>6d}')
    all_data.append({"문장": sent, "GPT-4": g, "방법1": v1, "방법3": v3, "방법1+3": vc})

print(f'\n{"합계":>40s} ', end="")
for key in ["GPT-4", "방법1", "방법3", "방법1+3"]:
    print(f'{sum(d[key] for d in all_data):>6d} ', end="")
print()

# 상세 토큰 확인 (읽을 수 있는 형태)
print(f'\n{"="*70}')
print("상세 토큰 분해 (읽기 변환)")
print(f'{"="*70}')
for sent in test_sentences[:2]:
    print(f'\n  "{sent}"')
    g_ids = enc.encode(sent)
    print(f'  {"GPT-4":>12s}: {[safe_gpt4(t) for t in g_ids]}')
    print(f'  {"방법1":>12s}: {readable_tokens(tok_v1.encode(sent).tokens)}')
    print(f'  {"방법3":>12s}: {readable_tokens(tok_v3.encode(sent).tokens)}')
    print(f'  {"방법1+3":>12s}: {readable_tokens(tok_combined.encode(sent).tokens)}')

In [ ]:
# ── 토큰 분해 상세 비교 ──

def safe_gpt4(t):
    raw = enc.decode_single_token_bytes(t)
    try:    return raw.decode("utf-8")
    except: return f"[{raw.hex()}]"

text = "인공지능이 자연어 처리 기술을 발전시키고 있다"
print(f'원문: "{text}"\n')

gpt4_t = enc.encode(text)
v1_t = tok_v1.encode(text)
v3_t = tok_v3.encode(text)
vc_t = tok_combined.encode(text)

rows = [
    ("GPT-4 BBPE",     [safe_gpt4(t) for t in gpt4_t]),
    ("방법1 (말뭉치)",   readable_tokens(v1_t.tokens)),
    ("방법3 (초기어휘)",  readable_tokens(v3_t.tokens)),
    ("방법1+3 (조합)",   readable_tokens(vc_t.tokens)),
]

for name, tokens in rows:
    broken = sum(1 for t in tokens if t.startswith("["))
    status = f"⚠ {broken}개 깨짐" if broken else "✓ 깨짐 없음"
    print(f"{name:>18s} ({len(tokens):>2d}토큰) {status}")
    print(f"{'':>18s}  {tokens}\n")

## 6. 실제 한국어 LLM 토크나이저 비교

HuggingFace에서 실제 한국어 LLM의 토크나이저를 불러와 비교합니다.

In [ ]:
from transformers import AutoTokenizer

# 실제 한국어 지원 LLM 토크나이저
tokenizers_info = {
    "GPT-4 (tiktoken)": None,  # 별도 처리
    "mBERT (WordPiece)": "bert-base-multilingual-cased",
    "XLM-R (SPM-BPE)": "xlm-roberta-base",
    "mT5 (SPM-Unigram)": "google/mt5-small",
}

loaded = {}
for name, model_id in tokenizers_info.items():
    if model_id:
        try:
            loaded[name] = AutoTokenizer.from_pretrained(model_id)
            print(f"✓ {name}: vocab={loaded[name].vocab_size}")
        except Exception as e:
            print(f"✗ {name}: {e}")

print(f"✓ GPT-4 (tiktoken): vocab=~100k")

In [ ]:
# ── 모든 토크나이저 + 우리가 만든 1+3 조합 비교 ──

test_sentences = [
    "인공지능이 세상을 바꾸고 있다",
    "자연어 처리는 놀랍습니다",
    "Transformer uses self-attention",
    "GPT-4는 BPE를 사용한다 🤖",
]

for sent in test_sentences:
    print(f'\n"{sent}"')
    print("-" * 60)
    
    # GPT-4
    g = enc.encode(sent)
    print(f"  {'GPT-4 (BBPE)':>20s}: {len(g):>2d}토큰  {[safe_gpt4(t) for t in g]}")
    
    # HuggingFace 토크나이저
    for name, tok in loaded.items():
        tokens = tok.tokenize(sent)
        print(f"  {name:>20s}: {len(tokens):>2d}토큰  {tokens}")
    
    # 우리의 방법 1+3
    vc = tok_combined.encode(sent)
    print(f"  {'우리 방법1+3':>20s}: {len(vc.tokens):>2d}토큰  {readable_tokens(vc.tokens)}")

## 7. 시각화 — 토큰 수 비교 차트

In [ ]:
# 한글 폰트 설정 (Colab 호환)
!apt-get -qq install -y fonts-nanum > /dev/null 2>&1 || true

import matplotlib
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

fm._load_fontmanager(try_read_cache=False)
for f in fm.findSystemFonts():
    try:
        name = fm.FontProperties(fname=f).get_name()
        if 'Nanum' in name or 'Apple' in name:
            matplotlib.rcParams['font.family'] = name
            break
    except:
        continue
matplotlib.rcParams['axes.unicode_minus'] = False

In [ ]:
ko_sentences = [
    "인공지능이 세상을 바꾸고 있다",
    "자연어 처리는 놀랍습니다",
    "딥러닝 모델은 대규모 데이터에서 학습한다",
    "한국어는 교착어로 조사와 어미가 결합한다",
]

method_names = ["GPT-4\n(BBPE)", "mBERT\n(WordPiece)", "XLM-R\n(SPM-BPE)", "mT5\n(SPM-Uni)", "방법1+3\n(ours)"]
colors = ["#d62728", "#1a4e7a", "#2ca02c", "#9467bd", "#e8862a"]

fig, ax = plt.subplots(figsize=(14, 6))
width = 0.15
x = range(len(ko_sentences))

for i, (mname, color) in enumerate(zip(method_names, colors)):
    values = []
    for sent in ko_sentences:
        if i == 0:  values.append(len(enc.encode(sent)))
        elif i == 4: values.append(len(tok_combined.encode(sent).tokens))
        else:
            tok_name = list(loaded.keys())[i - 1]
            values.append(len(loaded[tok_name].tokenize(sent)))
    
    bars = ax.bar([xi + width * i for xi in x], values, width,
                  label=mname.replace('\n', ' '), color=color, edgecolor="white")
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                str(int(bar.get_height())), ha="center", fontsize=8, fontweight="bold")

labels = [s[:16] + "..." if len(s) > 16 else s for s in ko_sentences]
ax.set_xticks([xi + width * 2 for xi in x])
ax.set_xticklabels(labels, fontsize=10)
ax.set_ylabel("토큰 수")
ax.set_title("한국어 문장의 토크나이저별 토큰 수 비교", fontweight="bold", fontsize=13)
ax.legend(loc="upper left", fontsize=9)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
plt.show()

## 정리

| 방법 | 원리 | 장점 | 단점 |
|------|------|------|------|
| **방법 1**: 한국어 말뭉치 학습 | 빈도 높은 한글 바이트 쌍이 자연 병합 | 가장 자연스러움 | 대규모 말뭉치 필요 |
| **방법 2**: 형태소 전처리 | 의미 단위로 미리 분리 후 BBPE | 기존 모델 재활용 | 파이프라인 복잡 |
| **방법 3**: 한글 초기 어휘 | 음절 11,172자를 base vocab에 추가 | 한글 절대 안 깨짐 | vocab 크기 증가 |
| **방법 1+3 조합** | 초기 어휘 + 대규모 학습 | **최적 효율** | 학습 비용 |

### 실제 한국어 LLM의 선택

| 모델 | 토크나이저 | 전략 |
|------|-----------|------|
| **KoGPT** (카카오) | BPE | 한국어 말뭉치 + 한글 초기 어휘 |
| **EXAONE** (LG) | BPE | 한/영 균형 말뭉치 + 확장 어휘 |
| **SOLAR** (업스테이지) | Llama 토크나이저 확장 | 영어 BPE + 한글 토큰 32K 추가 |
| **Qwen-2.5** (알리바바) | BPE | 중/영/한 다국어 대규모 학습 |